# FCC Fe (Austenite) EBSD - ML Dataset Generator  v2

Two independent cells:

| Cell | Purpose | Must run first? |
|------|---------|----------------|
| **1 — Dataset generator** | Builds master patterns, simulates all Kikuchi patterns, writes HDF5 | Yes |
| **2 — QC figures** | Reads HDF5, writes 5 diagnostic PNGs to `figures_v2/` | After cell 1 |


In [2]:
"""
FCC Fe (Austenite) EBSD — ML Dataset Generator  v2
====================================================
Generates a labelled HDF5 dataset of simulated Kikuchi patterns for
strain classification and regression.


Requirements
------------
    pip install kikuchipy diffsims orix diffpy.structure h5py numpy \
                matplotlib joblib tqdm

"""
"""
FCC Fe (Austenite) EBSD — ML Training Dataset Generator  v2
=============================================================
Generates a labelled EBSD pattern dataset for strain classification
AND regression.

Pipeline:
    1. Build kinematical master patterns for each strain level 
    2. Sample orientations uniformly from the FCC fundamental zone
    3. Project patterns onto a virtual detector per orientation × strain × noise
    4. Apply dynamic background removal
    5. Store as uint8 to HDF5 with embedded train/val/test split indices

HDF5 structure:
    /patterns           uint8    (N, H, W)       
    /strain_pct         float32  (N,)            
    /strain_class       uint8    (N,)          
    /euler_angles_deg   float32  (N, 3)         
    /lattice_params     float32  (N, 3)          a, b, c in Å
    /noise_level        float32  (N,)
    /deform_gradient    float32  (N, 9)         
    /split              str      (N,)            "train" | "val" | "test"
    /idx_train          int64    (n_train,)
    /idx_val            int64    (n_val,)
    /idx_test           int64    (n_test,)

Requirements:
    pip install kikuchipy diffsims orix diffpy.structure h5py numpy matplotlib joblib tqdm

References:
    Wilkinson et al., Ultramicroscopy 106:307 (2006)
    Marquardt et al., Ultramicroscopy 184:167 (2018)
    Rowenhorst et al., Modelling Simul. Mater. Sci. Eng. 23 (2015)
"""

import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from tqdm import tqdm

import kikuchipy as kp
from diffsims.crystallography import ReciprocalLatticeVector
from orix.crystal_map import Phase
from orix.quaternion import Rotation
from orix.sampling import get_sample_fundamental
from diffpy.structure import Atom, Lattice, Structure

# Parallel master-pattern building
try:
    from joblib import Parallel, delayed
    _JOBLIB = True
except ImportError:
    _JOBLIB = False
    print("[INFO] joblib not found — master patterns will be built sequentially.")


# =============================================================================
# Configuration  
# =============================================================================

CFG = {
    # ── Paths ─────────────────────────────────────────────────────────────────
    "project_dir":           None,   # ← set to a string to override auto-detection
    "output_subdir":         "ml_dataset_v2",

    # ── Crystal — FCC Fe (austenite), space group Fm-3m ──────────────────────
    "a0":                    3.59,       # Å, reference lattice parameter
    "poisson":               0.29,       # Poisson ratio
    "space_group":           225,

    # ── Simulation ───────────────────────────────────────────────────────────
    "voltage_kev":           20,
    "min_dspacing":          0.7,        # Å

    #  PATTERN SIZE 
    "master_size":           512,        # px  (was 256)
    "detector_shape":        (180, 180), # px  (was (80, 80))

    "pc":                    (0.42, 0.22, 0.50),  # projection centre (Bruker)
    "sample_tilt":           70.0,       # degrees

    # ── Orientations ─────────────────────────────────────────────────────────
    "n_orientations":        500,
    "resolution":            1.5,        # degrees — cubochoric sampling

    # ── Training strain levels ────────────────────────────────────────────────
    "strain_max_pct":        50.0,
    "n_train_strain_levels": 20,

    # ── Test strain levels (2 per class, drawn from bin centres) ─────────────
    "test_strains_pct":      [2.0, 4.0, 7.5, 12.5, 20.0, 27.0, 37.5, 47.5],

    # ── Strain class bins (for the classification head) ───────────────────────
    # Boundaries [0,5,15,30,50] define 4 classes.
    "strain_class_edges":    [0.0, 5.0, 15.0, 30.0, 50.0],
    "strain_class_names":    ["VeryLow", "Low", "Medium", "High"],

    # ── Background removal ───────────────────────────────────────────────────
    "dynamic_bg_std":        8,          # Gaussian sigma (px)

    # ── Noise — Poisson shot noise + Gaussian read noise ─────────────────────
    # Poisson shot noise + Gaussian read noise at 4 levels (0.0, 0.01, 0.02, 0.05).
    "noise_levels":          [0.0, 0.01, 0.02, 0.05],
    "poisson_scale":         1e4,

    # ── Dataset split (by orientation — prevents data leakage) ───────────────
    "train_frac":            0.70,
    "val_frac":              0.15,

    # ── HDF5 storage ─────────────────────────────────────────────────────────
    # uint8 patterns: divide by 255 in the loader to recover float [0,1].
    # ~4× smaller file vs float32 with no perceptible quality loss.
    "store_uint8":           True,

    # Chunk size aligned to a typical DataLoader batch (32 patterns).
    "hdf5_chunk_size":       32,

    # ── Parallelism ──────────────────────────────────────────────────────────
    # Number of parallel workers for master-pattern generation.
    # Set to 1 to disable (useful on machines with limited RAM).
    "n_jobs":                4,

    # ── Reproducibility ──────────────────────────────────────────────────────
    "seed":                  42,
}

# =============================================================================
# Auto-detect project directory
# =============================================================================


def _resolve_project_dir(cfg_override):
    """Return the project root, auto-detected from the notebook location."""
    if cfg_override is not None:
        return os.path.abspath(cfg_override)

    # ── Running as a converted .py script ────────────────────────────────────
    try:
        here = os.path.abspath(__file__)
        return os.path.dirname(here)
    except NameError:
        pass

    # ── Fallback: wherever Python was launched from ───────────────────────────
    fallback = os.path.abspath(os.getcwd())
    print(f"[INFO] project_dir auto-detected as cwd: {fallback}")
    print("       Set CFG[\"project_dir\"] to override.")
    return fallback


PROJECT_DIR = _resolve_project_dir(CFG["project_dir"])

OUTDIR  = os.path.join(PROJECT_DIR, CFG["output_subdir"])
H5_PATH = os.path.join(OUTDIR, "ebsd_fcc_fe_v2.h5")
FIG_DIR = os.path.join(PROJECT_DIR, "figures_v2")

os.makedirs(OUTDIR,  exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"[paths] project : {PROJECT_DIR}")
print(f"[paths] output  : {OUTDIR}")
print(f"[paths] figures : {FIG_DIR}")

rng = np.random.default_rng(CFG["seed"])
A0  = CFG["a0"]
NU  = CFG["poisson"]


# =============================================================================
# Crystal helpers
# =============================================================================

def make_phase(a, b=None, c=None, name="austenite"):
    """Build an orix Phase for the given (strained) lattice."""
    b = a if b is None else b
    c = a if c is None else c
    return Phase(
        name=name,
        space_group=CFG["space_group"],
        structure=Structure(
            atoms=[Atom("Fe", [0, 0, 0])],
            lattice=Lattice(a, b, c, 90, 90, 90),
        ),
    )


def strained_abc(eps_pct):
    """Lattice parameters (a, b, c) in Å for uniaxial strain along [100]."""
    eps = eps_pct / 100.0
    return A0 * (1.0 + eps), A0 * (1.0 - NU * eps), A0 * (1.0 - NU * eps)


def deformation_gradient(eps_pct):
    """3×3 deformation gradient tensor F = diag(1+ε, 1−νε, 1−νε)."""
    eps = eps_pct / 100.0
    return np.diag([1 + eps, 1 - NU * eps, 1 - NU * eps]).astype(np.float32)


def strain_to_class(eps_pct):
    """Return integer class index (0–3) for a given strain percentage."""
    edges = CFG["strain_class_edges"]
    for i, (lo, hi) in enumerate(zip(edges, edges[1:])):
        if lo <= eps_pct <= hi:
            return i
    return len(edges) - 2  # clamp to last class


# =============================================================================
# Master pattern (cached per strain level)
# =============================================================================

def build_master_pattern(phase):
    """
    Kinematical master pattern in Lambert projection.

    Using master_size=512 (vs 256 in v1) doubles the angular resolution of
    each Kikuchi band edge, which benefits high-resolution detectors.
    """
    rlv = ReciprocalLatticeVector.from_min_dspacing(
        phase, min_dspacing=CFG["min_dspacing"]
    )
    rlv.sanitise_phase()
    rlv.calculate_structure_factor()
    rlv.calculate_theta(voltage=CFG["voltage_kev"] * 1e3)
    rlv = rlv[rlv.allowed]
    sim = kp.simulations.KikuchiPatternSimulator(rlv)
    return sim.calculate_master_pattern(half_size=CFG["master_size"]).as_lambert()


detector = kp.detectors.EBSDDetector(
    shape=CFG["detector_shape"],
    pc=CFG["pc"],
    sample_tilt=CFG["sample_tilt"],
)

_mp_cache: dict = {}


def _build_and_cache(eps):
    """Helper for parallel master-pattern building — returns (eps, mp)."""
    a, b, c = strained_abc(eps)
    mp = build_master_pattern(
        make_phase(a, b, c, name=f"austenite_{eps:.4f}pct")
    )
    return eps, mp


def build_all_master_patterns(all_levels):
    """
    Build master patterns for every strain level, optionally in parallel.

    v1 built patterns sequentially inside the loop; v2 uses joblib.Parallel
    to cut wall-clock time by ~(n_jobs)× on multi-core machines.
    """
    if _JOBLIB and CFG["n_jobs"] > 1:
        print(f"  Using {CFG['n_jobs']} parallel workers...")
        results = Parallel(n_jobs=CFG["n_jobs"], prefer="threads")(
            delayed(_build_and_cache)(eps) for eps in all_levels
        )
    else:
        results = [_build_and_cache(eps) for eps in tqdm(all_levels, desc="  Master patterns")]

    for eps, mp in results:
        a, b, _ = strained_abc(eps)
        _mp_cache[eps] = mp
        print(f"  ε={eps:6.2f}%  a={a:.4f}  b={strained_abc(eps)[1]:.4f} Å")


# =============================================================================
# Pattern simulation
# =============================================================================

def remove_background(sig):
    """Dynamic background removal via Gaussian subtraction."""
    sig.remove_dynamic_background(
        operation="subtract",
        std=CFG["dynamic_bg_std"],
        show_progressbar=False,
    )
    return sig


def get_clean_pattern(eps_pct, rotation):
    """
    Simulate a background-corrected EBSD pattern.

    Returns
    -------
    pat : float32 ndarray, shape (H, W), normalised to [0, 1]
    """
    sig = _mp_cache[eps_pct].get_patterns(
        rotations=rotation,
        detector=detector,
        energy=CFG["voltage_kev"],
        compute=True,
        show_progressbar=False,
    )

    try:
        sig = remove_background(sig)
    except Exception as e:
        print(f"  [WARN] Background removal skipped at ε={eps_pct:.2f}%: {e}")

    pat = sig.data[0].astype(np.float32)
    return (pat - pat.min()) / (pat.max() - pat.min() + 1e-9)


def add_noise(pat, sigma, rng):
    """
    Two-component noise model: Poisson shot noise + Gaussian read noise.

    Returns
    -------
    float32 ndarray clipped to [0, 1]
    """
    if sigma == 0.0:
        return pat.copy()
    counts  = pat * CFG["poisson_scale"]
    poisson = rng.normal(0, np.sqrt(np.maximum(counts, 1e-6))) / CFG["poisson_scale"]
    gauss   = rng.normal(0, sigma, pat.shape).astype(np.float32)
    return np.clip(pat + poisson + gauss, 0, 1).astype(np.float32)


def float_to_uint8(pat):
    """Convert float32 [0,1] pattern to uint8 [0,255] for compact HDF5 storage."""
    return (pat * 255).round().astype(np.uint8)


# =============================================================================
# Orientation sampling
# =============================================================================

def sample_orientations(n, resolution_deg, seed):
    """
    Sample n rotations uniformly from the FCC (m-3m) fundamental zone
    using cubochoric sampling via orix.
    """
    from orix.quaternion.symmetry import get_point_group
    pg   = get_point_group(225, proper=False)
    full = get_sample_fundamental(
        resolution=resolution_deg, point_group=pg, method="cubochoric"
    )
    n_full    = full.size
    rng_local = np.random.default_rng(seed)
    if n_full >= n:
        idx = rng_local.choice(n_full, size=n, replace=False)
    else:
        print(f"  [WARN] Only {n_full} orientations at {resolution_deg}°; using all.")
        idx = np.arange(n_full)
    sampled = full[idx]
    print(f"  Orientations: {sampled.size} sampled from {n_full} in fundamental zone")
    return sampled


# =============================================================================
# Train / val / test split
# =============================================================================

def assign_splits(n_ori, train_f, val_f, seed):
    """
    Assign orientations to train/val/test by index shuffle.
    Splitting by orientation prevents data leakage across splits.
    """
    rng_local = np.random.default_rng(seed)
    idx     = rng_local.permutation(n_ori)
    n_train = int(n_ori * train_f)
    n_val   = int(n_ori * val_f)
    n_test  = n_ori - n_train - n_val
    splits  = {}
    for i, ori_i in enumerate(idx):
        if i < n_train:
            splits[ori_i] = "train"
        elif i < n_train + n_val:
            splits[ori_i] = "val"
        else:
            splits[ori_i] = "test"
    print(f"  Split: {n_train} train / {n_val} val / {n_test} test orientations")
    return splits


# =============================================================================
# Strain levels
# =============================================================================

def get_train_strain_levels():
    """
    Evenly-spaced training strain levels as Python floats.
    Float precision ensures exact cache key matches.
    v2 uses 30 levels (vs 20) for finer regression granularity.
    """
    n  = CFG["n_train_strain_levels"]
    mx = CFG["strain_max_pct"]
    return [round(float(v), 6) for v in np.linspace(0.0, mx, n)]


# =============================================================================
# Dataset generation
# =============================================================================

def generate_dataset():
    t0 = time.time()

    train_levels = get_train_strain_levels()
    test_levels  = [float(s) for s in CFG["test_strains_pct"]]
    all_levels   = sorted(set(train_levels + test_levels))

    # ── Step 1: Build (parallel) master patterns ──────────────────────────────
    print(f"\n[1/5] Building master patterns for {len(all_levels)} strain levels...")
    print(f"      master_size={CFG['master_size']} px  "
          f"detector={CFG['detector_shape'][0]}×{CFG['detector_shape'][1]} px")
    build_all_master_patterns(all_levels)
    print(f"  Done — {len(_mp_cache)} patterns cached ({time.time()-t0:.0f}s)")

    # ── Step 2: Sample orientations ───────────────────────────────────────────
    print("\n[2/5] Sampling orientations...")
    rotations = sample_orientations(CFG["n_orientations"], CFG["resolution"], CFG["seed"])
    n_ori     = rotations.size
    splits    = assign_splits(n_ori, CFG["train_frac"], CFG["val_frac"], CFG["seed"])

    n_strain       = len(train_levels)
    n_noise        = len(CFG["noise_levels"])
    n_noise_test   = 2
    n_test_strains = len(CFG["test_strains_pct"])
    n_train_ori    = sum(1 for v in splits.values() if v == "train")
    n_val_ori      = sum(1 for v in splits.values() if v == "val")
    n_test_ori     = sum(1 for v in splits.values() if v == "test")

    n_train = n_train_ori * n_strain * n_noise
    n_val   = n_val_ori   * n_strain * n_noise
    n_test  = n_test_ori  * n_test_strains * n_noise_test
    N_total = n_train + n_val + n_test
    H, W    = CFG["detector_shape"]

    # ── Step 3: Preview ───────────────────────────────────────────────────────
    print(f"\n[3/5] Dataset preview:")
    print(f"  Train : {n_train:>7}  ({n_train_ori} ori × {n_strain} strains × {n_noise} noise)")
    print(f"  Val   : {n_val:>7}  ({n_val_ori} ori × {n_strain} strains × {n_noise} noise)")
    print(f"  Test  : {n_test:>7}  ({n_test_ori} ori × {n_test_strains} strains × {n_noise_test} noise)")
    print(f"  Total : {N_total:>7}")
    dtype_str = "uint8" if CFG["store_uint8"] else "float32"
    est_mb = N_total * H * W * (1 if CFG["store_uint8"] else 4) / 1e6
    print(f"  Est. file size: {est_mb:.0f} MB ({dtype_str} patterns)")

    # ── Step 4: Simulate patterns ─────────────────────────────────────────────
    print(f"\n[4/5] Simulating {N_total} patterns...")

    pat_dtype = np.uint8 if CFG["store_uint8"] else np.float32

    all_patterns  = np.zeros((N_total, H, W), dtype=pat_dtype)
    all_strains   = np.zeros(N_total,         dtype=np.float32)
    all_classes   = np.zeros(N_total,         dtype=np.uint8)     # NEW
    all_eulers    = np.zeros((N_total, 3),    dtype=np.float32)
    all_lattice   = np.zeros((N_total, 3),    dtype=np.float32)
    all_defgrad   = np.zeros((N_total, 9),    dtype=np.float32)   # NEW
    all_noise     = np.zeros(N_total,         dtype=np.float32)
    all_splits    = np.empty(N_total,         dtype=object)
    idx_write     = 0

    for ori_i in tqdm(range(n_ori), desc="  Orientations", unit="ori"):
        rot         = rotations[ori_i]
        split_label = splits[ori_i]
        euler       = rot.to_euler(degrees=True).flatten()[:3]

        if split_label in ("train", "val"):
            strain_vals = train_levels
            noise_vals  = CFG["noise_levels"]
        else:
            strain_vals = test_levels
            noise_vals  = [0.0, 0.01]

        for eps_pct in strain_vals:
            a, b, c   = strained_abc(float(eps_pct))
            clean_pat = get_clean_pattern(float(eps_pct), rot)
            F_flat    = deformation_gradient(float(eps_pct)).flatten()
            cls       = strain_to_class(float(eps_pct))

            for sigma in noise_vals:
                noisy = add_noise(clean_pat, sigma, rng)
                all_patterns [idx_write] = float_to_uint8(noisy) if CFG["store_uint8"] else noisy
                all_strains  [idx_write] = eps_pct
                all_classes  [idx_write] = cls
                all_eulers   [idx_write] = euler
                all_lattice  [idx_write] = [a, b, c]
                all_defgrad  [idx_write] = F_flat
                all_noise    [idx_write] = sigma
                all_splits   [idx_write] = split_label
                idx_write += 1

    # Trim pre-allocated arrays to actual size (should already match N_total)
    all_patterns = all_patterns[:idx_write]
    all_strains  = all_strains [:idx_write]
    all_classes  = all_classes [:idx_write]
    all_eulers   = all_eulers  [:idx_write]
    all_lattice  = all_lattice [:idx_write]
    all_defgrad  = all_defgrad [:idx_write]
    all_noise    = all_noise   [:idx_write]
    all_splits   = all_splits  [:idx_write]

    # ── Step 5: Write HDF5 ────────────────────────────────────────────────────
    print(f"\n[5/5] Writing HDF5 → {H5_PATH}")
    chunk = (CFG["hdf5_chunk_size"], H, W)
    with h5py.File(H5_PATH, "w") as f:
        f.attrs["config"]      = json.dumps(CFG)
        f.attrs["generated"]   = time.strftime("%Y-%m-%dT%H:%M:%S")
        f.attrs["n_total"]     = idx_write
        f.attrs["n_train"]     = int(np.sum(all_splits == "train"))
        f.attrs["n_val"]       = int(np.sum(all_splits == "val"))
        f.attrs["n_test"]      = int(np.sum(all_splits == "test"))
        f.attrs["pattern_dtype"] = dtype_str
        f.attrs["uint8_scale"] = 255.0  # divide by this to get float32 [0,1]

        # Patterns: gzip-compressed, batch-aligned chunks
        f.create_dataset("patterns",         data=all_patterns,
                         compression="gzip", compression_opts=4,
                         chunks=chunk)
        f.create_dataset("strain_pct",       data=all_strains)
        f.create_dataset("strain_class",     data=all_classes)     # NEW
        f.create_dataset("euler_angles_deg", data=all_eulers)
        f.create_dataset("lattice_params",   data=all_lattice)
        f.create_dataset("deform_gradient",  data=all_defgrad)     # NEW
        f.create_dataset("noise_level",      data=all_noise)
        f.create_dataset("split",            data=all_splits.astype("S8"))

        for sp in ("train", "val", "test"):
            f.create_dataset(
                f"idx_{sp}",
                data=np.where(all_splits == sp)[0].astype(np.int64)
            )

    size_mb = os.path.getsize(H5_PATH) / 1e6
    print(f"  Saved {idx_write} patterns  ({size_mb:.1f} MB)")
    print(f"  Total time: {time.time()-t0:.0f}s")
# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":
    H_px, W_px = CFG["detector_shape"]
    print("=" * 60)
    print("  FCC Fe EBSD — ML Dataset Generator  v2")
    print(f"  Project  : {PROJECT_DIR}")
    print(f"  Output   : {OUTDIR}")
    print(f"  Ori.     : {CFG['n_orientations']}")
    print(f"  Strains  : {CFG['n_train_strain_levels']} levels (0–{CFG['strain_max_pct']}%)")
    print(f"  Noise    : {CFG['noise_levels']}")
    print(f"  Detector : {H_px}×{W_px} px  (master_size={CFG['master_size']})")
    print(f"  Storage  : {'uint8' if CFG['store_uint8'] else 'float32'}")
    print(f"  Seed     : {CFG['seed']}")
    print("=" * 60)

    generate_dataset()

    print("\n" + "=" * 60)
    print("  Dataset generation complete.")
    print(f"  Dataset : {H5_PATH}")
    print(f"  Run the QC cell below to generate figures.")
    print(f"  Run the Loader cell to write pytorch_dataset.py.")
    print("=" * 60)

[INFO] project_dir auto-detected as cwd: /Users/jfj3094/Documents/FP-465/MATSCI_465_local
       Set CFG["project_dir"] to override.
[paths] project : /Users/jfj3094/Documents/FP-465/MATSCI_465_local
[paths] output  : /Users/jfj3094/Documents/FP-465/MATSCI_465_local/ml_dataset_v2
[paths] figures : /Users/jfj3094/Documents/FP-465/MATSCI_465_local/figures_v2
  FCC Fe EBSD — ML Dataset Generator  v2
  Project  : /Users/jfj3094/Documents/FP-465/MATSCI_465_local
  Output   : /Users/jfj3094/Documents/FP-465/MATSCI_465_local/ml_dataset_v2
  Ori.     : 500
  Strains  : 20 levels (0–50.0%)
  Noise    : [0.0, 0.01, 0.02, 0.05]
  Detector : 120×120 px  (master_size=512)
  Storage  : uint8
  Seed     : 42

[1/5] Building master patterns for 28 strain levels...
      master_size=512 px  detector=120×120 px
  Using 4 parallel workers...


  0%|                                                     | 0/1 [00:00<?, ?it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.93it/s]



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.88it/s]

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.69it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.30it/s]

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  6.35it/s]

  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.19s/it]


  0%|      

  ε=  0.00%  a=3.5900  b=3.5900 Å
  ε=  2.00%  a=3.6618  b=3.5692 Å
  ε=  2.63%  a=3.6845  b=3.5626 Å
  ε=  4.00%  a=3.7336  b=3.5484 Å
  ε=  5.26%  a=3.7789  b=3.5352 Å
  ε=  7.50%  a=3.8592  b=3.5119 Å
  ε=  7.89%  a=3.8734  b=3.5078 Å
  ε= 10.53%  a=3.9679  b=3.4804 Å
  ε= 12.50%  a=4.0388  b=3.4599 Å
  ε= 13.16%  a=4.0624  b=3.4530 Å
  ε= 15.79%  a=4.1568  b=3.4256 Å
  ε= 18.42%  a=4.2513  b=3.3982 Å
  ε= 20.00%  a=4.3080  b=3.3818 Å
  ε= 21.05%  a=4.3458  b=3.3708 Å
  ε= 23.68%  a=4.4403  b=3.3434 Å
  ε= 26.32%  a=4.5347  b=3.3160 Å
  ε= 27.00%  a=4.5593  b=3.3089 Å
  ε= 28.95%  a=4.6292  b=3.2886 Å
  ε= 31.58%  a=4.7237  b=3.2612 Å
  ε= 34.21%  a=4.8182  b=3.2338 Å
  ε= 36.84%  a=4.9126  b=3.2064 Å
  ε= 37.50%  a=4.9362  b=3.1996 Å
  ε= 39.47%  a=5.0071  b=3.1790 Å
  ε= 42.11%  a=5.1016  b=3.1516 Å
  ε= 44.74%  a=5.1961  b=3.1242 Å
  ε= 47.37%  a=5.2905  b=3.0968 Å
  ε= 47.50%  a=5.2953  b=3.0955 Å
  ε= 50.00%  a=5.3850  b=3.0694 Å
  Done — 28 patterns cached (14s)

[2/5] Samplin

  Orientations: 100%|████████████████████████| 500/500 [02:33<00:00,  3.26ori/s]



[5/5] Writing HDF5 → /Users/jfj3094/Documents/FP-465/MATSCI_465_local/ml_dataset_v2/ebsd_fcc_fe_v2.h5
  Saved 35200 patterns  (406.7 MB)
  Total time: 183s

  Dataset generation complete.
  Dataset : /Users/jfj3094/Documents/FP-465/MATSCI_465_local/ml_dataset_v2/ebsd_fcc_fe_v2.h5
  Run the QC cell below to generate figures.
  Run the Loader cell to write pytorch_dataset.py.


In [3]:
"""
QC Figures
==========
Reads the completed HDF5 dataset and writes five diagnostic figures
to <project_dir>/figures_v2/.

Run after the dataset generator cell.
"""
# =============================================================================
# QC figures
# =============================================================================

def make_qc_figures():
    print("\nGenerating QC figures...")

    with h5py.File(H5_PATH, "r") as f:
        strains      = f["strain_pct"][:]
        splits       = f["split"][:].astype(str)
        eulers       = f["euler_angles_deg"][:]
        noise        = f["noise_level"][:]
        idx_tr       = f["idx_train"][:]
        idx_va       = f["idx_val"][:]
        idx_te       = f["idx_test"][:]
        n_ex         = min(3, len(idx_tr))
        ex_pats_raw  = f["patterns"][idx_tr[:n_ex]]
        ex_str       = strains[idx_tr[:n_ex]]
        ex_ns        = noise[idx_tr[:n_ex]]
        test_pats_raw = f["patterns"][idx_te]
        test_strains  = strains[idx_te]
        uint8_scale   = f.attrs.get("uint8_scale", 1.0)

    # Decode uint8 → float32 for display
    def decode(raw):
        return raw.astype(np.float32) / uint8_scale if CFG["store_uint8"] else raw

    ex_pats    = decode(ex_pats_raw)
    test_pats  = decode(test_pats_raw)

    plt.rcParams.update({
        "font.family":       "Arial",
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.labelsize":    10,
        "axes.titlesize":    10,
        "axes.titleweight":  "bold",
        "xtick.labelsize":   9,
        "ytick.labelsize":   9,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linestyle":    "--",
    })

    BLUE   = "#2E74B5"
    ORANGE = "#C55A11"
    GREEN  = "#4EA72A"
    NAVY   = "#1F497D"
    PURPLE = "#7030A0"

    # ── Figure 1: QC summary ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 13))
    fig.suptitle("FCC Fe Austenite — EBSD ML Dataset Quality Control  (v2)",
                 fontsize=13, fontweight="bold", y=0.98)
    gs = GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38)

    # A: Training strain distribution
    ax_a = fig.add_subplot(gs[0, 0])
    ax_a.hist(strains[splits != "test"], bins=60, color=BLUE,
              edgecolor="white", linewidth=0.3)
    ax_a.set_xlabel("Strain (%)")
    ax_a.set_ylabel("Count")
    ax_a.set_title("A  Training Strain Distribution")
    ax_a.text(0.5, -0.28,
              f"Uniform sampling across 0–50%.\n"
              f"{CFG['n_train_strain_levels']} discrete levels, equal representation.",
              transform=ax_a.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    # B: Test set strain levels
    ax_b = fig.add_subplot(gs[0, 1])
    test_strain_vals = sorted(CFG["test_strains_pct"])
    test_counts = [
        int(np.sum(np.abs(strains[splits == "test"] - lvl) < 0.01))
        for lvl in test_strain_vals
    ]
    bars_b = ax_b.bar([f"{v:.1f}" for v in test_strain_vals], test_counts,
                      color=ORANGE, edgecolor="white")
    ax_b.set_xlabel("Strain (%)")
    ax_b.set_ylabel("Count")
    ax_b.tick_params(axis="x", labelsize=7.5, rotation=40)
    for bar, cnt in zip(bars_b, test_counts):
        ax_b.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                  str(cnt), ha="center", fontsize=7, fontweight="bold")
    ax_b.set_title("B  Test Set Strain Levels")

    # C: Dataset split sizes
    ax_c = fig.add_subplot(gs[0, 2])
    counts     = [len(idx_tr), len(idx_va), len(idx_te)]
    split_cols = [BLUE, GREEN, ORANGE]
    bars_c = ax_c.bar(["Train", "Val", "Test"], counts,
                      color=split_cols, edgecolor="white", width=0.5)
    for bar, cnt in zip(bars_c, counts):
        ax_c.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
                  f"{cnt:,}", ha="center", fontsize=9, fontweight="bold")
    ax_c.set_ylabel("Patterns")
    ax_c.set_title("C  Dataset Split Sizes")

    # D: Orientation coverage
    ax_d = fig.add_subplot(gs[1, 0])
    stride = max(1, CFG["n_train_strain_levels"] * len(CFG["noise_levels"]))
    tr_idx_unique = idx_tr[::stride]
    ax_d.scatter(eulers[tr_idx_unique, 0], eulers[tr_idx_unique, 1],
                 s=5, alpha=0.6, color=NAVY, linewidths=0)
    ax_d.set_xlabel("φ₁ (°)")
    ax_d.set_ylabel("Φ (°)")
    ax_d.set_xlim(0, 360)
    ax_d.set_ylim(0, 65)
    ax_d.set_title("D  Orientation Coverage (train)")

    # E: Noise level distribution — now 6 levels (was 4)
    ax_e = fig.add_subplot(gs[1, 1])
    unique_noise, noise_counts = np.unique(noise[splits == "train"], return_counts=True)
    ax_e.bar([f"{v:.2f}" for v in unique_noise], noise_counts,
             color="#4472C4", edgecolor="white", width=0.6)
    ax_e.set_xlabel("Gaussian σ")
    ax_e.set_ylabel("Count")
    ax_e.set_title("E  Noise Level Distribution (train)")
    ax_e.text(0.5, -0.28,
              f"{len(CFG['noise_levels'])} noise tiers (0–{max(CFG['noise_levels'])}).\n"
              "Improves model robustness to detector read-noise.",
              transform=ax_e.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    # F: Example patterns
    captions = ["Clean (σ=0.00)", "Light noise (σ=0.01)", "Moderate noise (σ=0.02)"]
    for i in range(n_ex):
        ax = fig.add_subplot(gs[2, i])
        ax.imshow(ex_pats[i], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"ε = {ex_str[i]:.1f}%  |  σ = {ex_ns[i]:.2f}", fontsize=9, pad=4)
        ax.axis("off")
        ax.text(0.5, -0.10, captions[i], transform=ax.transAxes,
                ha="center", va="top", fontsize=8, color="#444", style="italic")

    H_px, W_px = CFG["detector_shape"]
    fig.text(0.5, 0.005,
             f"Figure F — Example EBSD Kikuchi patterns ({H_px}×{W_px} px) after dynamic background removal. "
             "Bands remain visible across all noise levels.",
             ha="center", fontsize=8.5, color="#333", style="italic")

    fig.savefig(os.path.join(FIG_DIR, "dataset_qc.png"), dpi=150, bbox_inches="tight")
    print("  Saved → dataset_qc.png")
    plt.close(fig)

    # ── Figure 2: Patterns at each test strain level ──────────────────────────
    test_levels_sorted = sorted(CFG["test_strains_pct"])
    level_pats  = {}
    for lvl in test_levels_sorted:
        mask = (np.abs(test_strains - lvl) < 0.01) & (noise[idx_te] == 0.0)
        if not np.any(mask):
            mask = np.abs(test_strains - lvl) == np.abs(test_strains - lvl).min()
        level_pats[lvl] = test_pats[np.where(mask)[0][0]]

    bin_edges   = CFG["strain_class_edges"]
    class_names = CFG["strain_class_names"]
    class_cols  = {"VeryLow": BLUE, "Low": GREEN, "Medium": ORANGE, "High": PURPLE}

    def get_class_name(eps):
        for ci, (lo, hi) in enumerate(zip(bin_edges, bin_edges[1:])):
            if lo <= eps <= hi:
                return class_names[ci]
        return ""

    n_levels = len(test_levels_sorted)
    fig2, axes2 = plt.subplots(2, n_levels, figsize=(2.8 * n_levels, 6),
                                gridspec_kw={"height_ratios": [1, 0.08]})
    fig2.suptitle("EBSD Kikuchi Patterns — Test Set (Clean)  [v2: 120×120 px]",
                  fontsize=11, fontweight="bold", y=1.01)

    for j, lvl in enumerate(test_levels_sorted):
        ax_img = axes2[0][j]
        ax_lbl = axes2[1][j]
        ax_img.imshow(level_pats[lvl], cmap="gray", vmin=0, vmax=1)
        ax_img.set_title(f"ε = {lvl:.1f}%", fontsize=9, pad=3)
        ax_img.axis("off")
        cls = get_class_name(lvl)
        ax_lbl.text(0.5, 0.5, cls, transform=ax_lbl.transAxes,
                    ha="center", va="center", fontsize=8,
                    color="white", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3",
                              facecolor=class_cols.get(cls, "gray"),
                              edgecolor="none"))
        ax_lbl.axis("off")

    fig2.tight_layout()
    fig2.savefig(os.path.join(FIG_DIR, "patterns_all_strains.png"),
                 dpi=150, bbox_inches="tight")
    print("  Saved → patterns_all_strains.png")
    plt.close(fig2)

    # ── Figure 3: Difference maps ─────────────────────────────────────────────
    ref_pat  = level_pats[test_levels_sorted[0]]
    non_zero = [lvl for lvl in test_levels_sorted if lvl > 0]
    n_diff   = len(non_zero)
    vmax_g   = max(np.abs(level_pats[lvl] - ref_pat).max() for lvl in non_zero)

    fig3, axes3 = plt.subplots(1, n_diff, figsize=(2.8 * n_diff, 4.5))
    fig3.suptitle("Kikuchi Band Shift Maps — Strained vs. Reference (ε = 0%)",
                  fontsize=11, fontweight="bold")

    for ax, lvl in zip(axes3, non_zero):
        dm  = level_pats[lvl] - ref_pat
        im  = ax.imshow(dm, cmap="RdBu_r", vmin=-vmax_g, vmax=vmax_g)
        mad = np.mean(np.abs(dm))
        ax.set_title(f"ε = {lvl:.1f}%", fontsize=9, pad=3)
        ax.text(0.5, -0.06, f"MAD = {mad:.4f}",
                transform=ax.transAxes, ha="center", fontsize=8, color="#444")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig3.tight_layout()
    fig3.savefig(os.path.join(FIG_DIR, "diff_maps.png"), dpi=150, bbox_inches="tight")
    print("  Saved → diff_maps.png")
    plt.close(fig3)

    # ── Figure 4: Strain sensitivity ──────────────────────────────────────────
    mad_vals     = [np.mean(np.abs(level_pats[lvl] - ref_pat)) for lvl in non_zero]
    class_colors = ["#D6E8F7", "#D5ECD5", "#FDE8D8", "#EAD6F5"]
    class_labels = ["Very Low\n(0–5%)", "Low\n(5–15%)", "Medium\n(15–30%)", "High\n(30–50%)"]

    fig4, ax4 = plt.subplots(figsize=(8, 4.5))
    ax4.plot(non_zero, mad_vals, "o-", color=NAVY,
             linewidth=2, markersize=7, markerfacecolor=BLUE, markeredgecolor="white")
    for eps, mad in zip(non_zero, mad_vals):
        ax4.annotate(f"{mad:.4f}", (eps, mad),
                     textcoords="offset points", xytext=(6, 5), fontsize=8)
    for (lo, hi), col, lbl in zip(zip(bin_edges, bin_edges[1:]), class_colors, class_labels):
        ax4.axvspan(lo, hi, alpha=0.35, color=col, zorder=0)
        ax4.text((lo + hi) / 2, 0, lbl, ha="center", va="bottom",
                 fontsize=7.5, color="#555")
    ax4.set_xlabel("Applied Strain ε (%)", fontsize=11)
    ax4.set_ylabel("Mean Absolute Difference (MAD)", fontsize=11)
    ax4.set_title("EBSD Pattern Sensitivity to Strain", fontsize=11, fontweight="bold")
    ax4.set_xlim(0, 52)
    fig4.tight_layout()
    fig4.savefig(os.path.join(FIG_DIR, "strain_sensitivity.png"),
                 dpi=150, bbox_inches="tight")
    print("  Saved → strain_sensitivity.png")
    plt.close(fig4)

    # ── Figure 5 (NEW): All-noise montage for a single orientation ────────────
    # Show the same pattern at every noise level side by side.
    n_noise_lvls = len(CFG["noise_levels"])
    # Find a clean train pattern and regenerate it at each noise level on-the-fly
    fig5, axes5 = plt.subplots(1, n_noise_lvls, figsize=(2.5 * n_noise_lvls, 3.5))
    fig5.suptitle("Same Pattern at Every Noise Level", fontsize=11, fontweight="bold")

    # Re-use the first example pattern; add noise post-hoc for display only
    rng_disp = np.random.default_rng(0)
    base_pat = ex_pats[0]
    for ax, sigma in zip(axes5, CFG["noise_levels"]):
        disp = add_noise(base_pat, sigma, rng_disp)
        ax.imshow(disp, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"σ = {sigma:.2f}", fontsize=9)
        ax.axis("off")
    fig5.tight_layout()
    fig5.savefig(os.path.join(FIG_DIR, "noise_montage.png"),
                 dpi=150, bbox_inches="tight")
    print("  Saved → noise_montage.png")
    plt.close(fig5)

    plt.rcParams.update(plt.rcParamsDefault)

# ── Run ───────────────────────────────────────────────────────────────────────
make_qc_figures()
print("\nAll QC figures saved to:", FIG_DIR)


Generating QC figures...
  Saved → dataset_qc.png
  Saved → patterns_all_strains.png
  Saved → diff_maps.png
  Saved → strain_sensitivity.png
  Saved → noise_montage.png

All QC figures saved to: /Users/jfj3094/Documents/FP-465/MATSCI_465_local/figures_v2
